# PID vs LPV H-inf 성능 비교

PID-FF와 LPV H-infinity 제어기의 경로 추종 성능을 다양한 속도에서 비교합니다.

This notebook compares the path-following performance of a PID feedforward controller against an LPV H-infinity controller across four speeds (0.6, 1.0, 1.5, 2.0 m/s).

**Author:** Suwon Lee, Kookmin University

In [ ]:
# -*- coding: utf-8 -*-
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from vfg_pathfollowing import Simulator, StepCurvaturePath, compute_metrics

## 1. 경로 및 속도 설정

Step curvature 경로 (R=0.5m, 90도 원호)에서 4가지 속도로 비교합니다. `L2=30`으로 설정하여 정상 상태 구간을 충분히 확보합니다.

In [ ]:
# 경로 생성 (충분한 직선 구간 확보)
path = StepCurvaturePath(R=0.5, theta_arc=np.pi/2, L2=30)

# 비교할 속도 목록
speeds = [0.6, 1.0, 1.5, 2.0]

# 속도별 시뮬레이션 실행 및 비교 플롯
all_results = {}
for v in speeds:
    sim = Simulator(path, speed=v)
    results = sim.compare(controllers=['lpv-hinf', 'pid-ff'], T=25.0)
    all_results[v] = results
    
    print(f"--- v = {v} m/s ---")
    Simulator.plot_comparison(results, path=path)
    plt.suptitle(f'v = {v} m/s', fontsize=12, y=1.02)
    plt.show()

## 2. 성능 지표 테이블

각 속도와 제어기 조합에 대한 RMS 방위각 오차와 정착 시간을 정리합니다.

In [ ]:
# 성능 지표 테이블 출력
print(f"{'Speed':>7s} | {'Controller':>10s} | {'RMS e_psi [deg]':>16s} | {'Max e_psi [deg]':>16s} | {'Settling [s]':>13s}")
print("-" * 75)

metrics_table = {}  # (speed, ctrl_key) -> metrics
for v in speeds:
    for key in ['lpv-hinf', 'pid-ff']:
        m = compute_metrics(all_results[v][key])
        metrics_table[(v, key)] = m
        label = 'LPV H-inf' if key == 'lpv-hinf' else 'PID-FF'
        print(f"{v:>6.1f}  | {label:>10s} | {m['rms_e_psi_deg']:>16.3f} | {m['max_e_psi_deg']:>16.3f} | {m['settling_time']:>13.2f}")

## 3. RMS 오차 막대 그래프

속도별 RMS 방위각 오차를 막대 그래프로 비교합니다. 속도가 높아질수록 LPV H-inf 제어기의 우위가 뚜렷해지는 것을 확인할 수 있습니다.

In [ ]:
# RMS 방위각 오차 막대 그래프
x = np.arange(len(speeds))
width = 0.35

rms_lpv = [metrics_table[(v, 'lpv-hinf')]['rms_e_psi_deg'] for v in speeds]
rms_pid = [metrics_table[(v, 'pid-ff')]['rms_e_psi_deg'] for v in speeds]

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, rms_lpv, width, label='LPV H-inf', color='#1f77b4')
bars2 = ax.bar(x + width/2, rms_pid, width, label='PID-FF', color='#ff7f0e')

ax.set_xlabel('Speed [m/s]')
ax.set_ylabel('RMS e_psi [deg]')
ax.set_title('Steady-State RMS Heading Error by Speed')
ax.set_xticks(x)
ax.set_xticklabels([f'{v}' for v in speeds])
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 막대 위에 수치 표시
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.2f}',
            ha='center', va='bottom', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.2f}',
            ha='center', va='bottom', fontsize=9)

fig.tight_layout()
plt.show()